In [ ]:
#1) Define partitions once
# Somewhere central, like pendo/config.py or pendo/partitions.py:

from dataclasses import dataclass

@dataclass(frozen=True)
class Partition:
    name: str          # "SAE" / "CAE"
    host: str
    api_key: str
    app_prefix: str    # "SAE_" or "C/SAE_" etc.

PARTITIONS = [
    Partition(name="SAE", host=BASE_URL_SAE, api_key=API_KEY_SAE, app_prefix="SAE_"),
    Partition(name="CAE", host=BASE_URL_CAE, api_key=API_KEY_CAE, app_prefix="CAE_"),
]

In [ ]:
#2) Shared wrapper for any pull
# src/pendo/pulls/partitioned.py

from typing import Callable, Any
from pendo.client import PendoClient
from pendo.partitions import PARTITIONS

def make_clients() -> dict[str, tuple[PendoClient, str]]:
    return {
        p.name: (PendoClient(host=p.host, key=p.api_key), p.app_prefix)
        for p in PARTITIONS
    }

def add_app_key(row: dict, *, partition: str, app_prefix: str) -> None:
    app_id = row.get("appId")
    if app_id is None:
        return
    row["partition"] = partition
    row["app_key"] = f"{app_prefix}{app_id}"

def pull_across_partitions(pull_one: Callable[[PendoClient], list[dict]]) -> list[dict]:
    clients = make_clients()
    out: list[dict] = []

    for partition, (client, app_prefix) in clients.items():
        rows = pull_one(client)
        for r in rows:
            add_app_key(r, partition=partition, app_prefix=app_prefix)
        print(f"[pull] partition={partition} rows={len(rows)}")
        out.extend(rows)

    print(f"[pull] partitions total rows={len(out)}")
    return out

In [ ]:
# 1) Update src/extract.py to add a two-phase orchestrator
# Keep your existing extract_all() if you want, but add extract_all_two_phase() and use it from __main__.
# This keeps extract.py “orchestrator only.” No payloads. No request shapes.
# src/extract.py

from pendo.client import PendoClient
from pendo.config import BASE_URL, API_KEY
from pendo.pulls import PULL_FUNCTIONS
from pendo.pulls.ux_lite_poll_events import pull_ux_lite_poll_events_for_registry  # NEW


def make_client() -> PendoClient:
    return PendoClient(host=BASE_URL, key=API_KEY)


def extract_all_two_phase(client: PendoClient) -> dict[str, object]:
    """
    Orchestrator only.

    Phase 1: run standard pulls (guides, aggregations, etc.)
    Phase 2: run dependent pull(s) that require phase-1 outputs
    """
    raw: dict[str, object] = {}

    # ---- Phase 1 ----
    for name, func in PULL_FUNCTIONS.items():
        # IMPORTANT: ux_lite_poll_events should NOT be in PULL_FUNCTIONS anymore (see section 3).
        print(f"[extract] pulling {name}...")
        raw[name] = func(client)

        try:
            n = len(raw[name])
            print(f"[extract] pulled {name}: {n}")
        except TypeError:
            print(f"[extract] pulled {name}")

        if name == "aggregations" and isinstance(raw[name], dict):
            results = raw[name].get("results", [])
            print(f"[extract] aggregations results rows: {len(results)}")
            if results:
                app_ids = {r.get("appId") for r in results if isinstance(r, dict) and "appId" in r}
                print(f"[extract] aggregations distinct appIds: {len(app_ids)}")

    # ---- Phase 2 (dependent) ----
    raw["ux_lite_poll_events"] = pull_ux_lite_poll_events_for_registry(
        client,
        guides=raw["guides"],
        aggregations=raw["aggregations"],
    )
    print(f"[extract] pulled ux_lite_poll_events (registry-filtered): {len(raw['ux_lite_poll_events'])}")

    return raw


if __name__ == "__main__":
    client = make_client()
    raw = extract_all_two_phase(client)
    print(f"Pulled guides: {len(raw['guides'])}")

In [ ]:
""""
2) Implement the dependent pull in pendo/pulls/ux_lite_poll_events.py

This function:
- builds the registry from guides + guideSeen aggregations (what you already decided is truth)
- extracts pollIds
- chunks pollEvents since cutoff
- filters rows to those pollIds
- Put all payloads here.

Yes, this imports build_ux_lite_registry. That’s okay here because:
- it guarantees the pollId list matches your strict UX-Lite definition
- it avoids duplicating your detection logic in two places (that’s the robust choice)
"""

# src/pendo/pulls/ux_lite_poll_events.py

from __future__ import annotations
from typing import Any, Dict, Iterable
import pandas as pd

from pendo.client import PendoClient
from pendo.endpoints import AggregationEndpoint
from pendo.util import get_endpoint

from transform.ux_lite import build_ux_lite_registry  # uses your strict template logic


CUTOFF_MS = 1761955200000  # 2025-11-01 00:00:00 UTC
CHUNK_DAYS = 30


def _pull_poll_events_chunk(client: PendoClient, *, first_ms: int, days: int) -> list[dict[str, Any]]:
    endpoint = get_endpoint("aggregations", client)
    assert isinstance(endpoint, AggregationEndpoint)

    payload: Dict[str, Any] = {
        "response": {"mimeType": "application/json"},
        "request": {
            "pipeline": [
                {
                    "source": {
                        "timeSeries": {"period": "dayRange", "first": first_ms, "count": -days},
                        "pollEvents": {"appId": "expandAppIds(\"*\")"},
                    }
                }
            ]
        },
    }

    resp = endpoint.run(payload)
    if isinstance(resp, dict):
        return resp.get("results", []) or []
    return []


def pull_ux_lite_poll_events_for_registry(
    client: PendoClient,
    *,
    guides: list[dict[str, Any]],
    aggregations: dict[str, Any],
    cutoff_ms: int = CUTOFF_MS,
    chunk_days: int = CHUNK_DAYS,
) -> list[dict[str, Any]]:
    guide_seen = pd.json_normalize((aggregations or {}).get("results", []))

    registry = build_ux_lite_registry(guides, guide_seen)

    poll_ids: set[str] = set()
    for col in ["pollId_usability", "pollId_usefulness", "pollId_comment"]:
        if col in registry.columns:
            poll_ids |= set(registry[col].dropna().astype(str).str.strip())

    print(f"[ux_lite_poll_events] registry rows={len(registry)} pollIds={len(poll_ids)}")

    now_ms = int(pd.Timestamp.utcnow().timestamp() * 1000)
    cursor = now_ms
    kept_all: list[dict[str, Any]] = []

    while cursor > cutoff_ms:
        chunk_start = max(cutoff_ms, cursor - chunk_days * 24 * 60 * 60 * 1000)

        rows = _pull_poll_events_chunk(client, first_ms=cursor, days=chunk_days)

        kept = []
        for r in rows:
            pid = r.get("pollId")
            if pid is None:
                continue
            if str(pid).strip() in poll_ids:
                kept.append(r)

        print(
            f"[ux_lite_poll_events] chunk {pd.to_datetime(chunk_start, unit='ms', utc=True).date()} → "
            f"{pd.to_datetime(cursor, unit='ms', utc=True).date()} rows={len(rows)} kept={len(kept)}"
        )

        kept_all.extend(kept)
        cursor = chunk_start

    return kept_all

In [ ]:
"""
3) Update pendo/pulls/__init__.py so ux_lite_poll_events is NOT in PULL_FUNCTIONS

Because phase 2 is now responsible for that.

So PULL_FUNCTIONS should be only “phase 1” pulls:
"""

PULL_FUNCTIONS = {
    "guides": pull_guides,
    "aggregations": pull_aggregations,
    # remove ux_lite_poll_events here
}

""""
4) What you should see after this

When you run main.py (or src/extract.py):

pollEvents should no longer be ~300 total

you should see multiple chunks printed

ux_lite_local_responses should jump above 73 and include more apps

If you paste your current pull_aggregations function signature and where it lives (just the name/path), I’ll make sure the dependent poll pull is using the same endpoint helper you already rely on, so it drops in cleanly.
"""

In [ ]:
import requests
requests.get("https://app.pendo.io")